Install and Load Required Packages

In [ ]:
# Run this in a Jupyter cell to install
!pip install polars
!pip install pandas numpy

import os
import requests
import pandas as pd
import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from pathlib import Path

List of OMOP files

omop-condition-era-export.csv        
omop-drug-era-export.csv     
omop-observation-export.csv         
omop-procedure-occurrence-export.csv
omop-condition-occurrence-export.csv  
omop-drug-exposure.csv       
omop-observation-period-export.csv  
omop-visit-detail-export.csv
omop-death-export.csv                 
omop-measurement-export.csv  
omop-person-export.csv              
omop-visit-occurrence-export.csv

In [ ]:
# Check OMOP file
import pandas as pd

file_path = '/choruspilot/mgh/omop-person-export.csv'

try:
    # Read ONLY the first 1000 rows
    df_sample = pd.read_csv(file_path)#, nrows=1000)
    
    print("Successfully loaded sample!")
    print(df_sample.info())
    display(df_sample.head())

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
# Load all tables into a dictionary for easy access
from omop_utils import set_verbose 
from omop_calc_sofa import compute_daily_sofa, compute_hourly_sofa
from omop_calc_sepsis3 import compute_suspected_infection, evaluate_sepsis3

import warnings
warnings.filterwarnings('ignore') # Silences the dateutil formatting warning

import pandas as pd
import numpy as np

# FIX: Grouped imports cleanly at the top
from omop_utils import set_verbose
from omop_calc_sofa import compute_daily_sofa
from omop_calc_sepsis3 import compute_suspected_infection, evaluate_sepsis3

file_mapping = {
    'person': 'omop-person-export',
    'visit_occurrence': 'omop-visit-occurrence-export',
    'measurement': 'omop-measurement-export',
    'drug_exposure': 'omop-drug-exposure',
    'procedure_occurrence': 'omop-procedure-occurrence-export',
    'condition_occurrence': 'omop-condition-occurrence-export'
}

cdm = {}
print("Loading tables...")

for standard_name, file_name in file_mapping.items():
    file_path = f'/choruspilot/mgh/{file_name}.csv'
    
    # Load with nrows for testing. Remove this later for the full run!
    df = pd.read_csv(file_path, nrows=500000)
    
#     # Force all column names to lowercase
#     df.columns = df.columns.str.lower()

#     # Parse as UTC, strip the timezone entirely, and FORCE nanosecond precision
#     for col in df.columns:
#         if 'date' in col or 'time' in col:
#             df[col] = pd.to_datetime(df[col], errors='coerce', utc=True).dt.tz_localize(None).astype('datetime64[ns]')

    cdm[standard_name] = df

# Handle the missing concept_ancestor safely
ancestor_df = pd.DataFrame() 



In [ ]:
# Summary of the OMOP data
from omop_utils import set_verbose
set_verbose(True)

person_df = cdm['person']
visit_df = cdm['visit_occurrence']
measurement_df = cdm['measurement']
drug_df = cdm['drug_exposure']
procedure_df = cdm['procedure_occurrence']
condition_df = cdm['condition_occurrence']

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Total patients in CDM: {len(person_df)}")
print(f"Total visits: {len(visit_df)}")
print(f"Total measurements: {len(measurement_df)}")
print(f"Total drug exposures: {len(drug_df)}")
print(f"Total procedures: {len(procedure_df)}")
print(f"Total condition: {len(condition_df)}")

 
# Check date range
if not visit_df.empty:
    visit_df['visit_start_datetime'] = pd.to_datetime(visit_df['visit_start_datetime'])
    print(f"Date range: {visit_df['visit_start_datetime'].min()} to {visit_df['visit_start_datetime'].max()}")
    print(f"Patients with ICU visits: {visit_df['person_id'].nunique()}")
 
# Check measurement types
if not measurement_df.empty:
    print(f"\nTop measurement concepts:")
    top_meas = measurement_df['measurement_concept_id'].value_counts().head(10)
    for cid, count in top_meas.items():
        print(f"  {cid}: {count} records")
 
# Check drug types  
if not drug_df.empty:
    print(f"\nTop drug concepts:")
    top_drugs = drug_df['drug_concept_id'].value_counts().head(10)
    for cid, count in top_drugs.items():
        print(f"  {cid}: {count} records")
 
print("="*60)
 
from omop_utils import print_dataset_summary
print_dataset_summary(cdm)

In [ ]:
# Analyze the data and find Sepsis-3 cases
set_verbose(True)

# Compute SOFA (hourly for accuracy, daily for reporting)
hourly_sofa = compute_hourly_sofa(cdm, ancestor_df)
daily_sofa = compute_daily_sofa(cdm, ancestor_df)

# Find infections
suspected = compute_suspected_infection(cdm, ancestor_df)

# Evaluate Sepsis-3 
sepsis3 = evaluate_sepsis3(hourly_sofa, suspected, cdm, ancestor_df)

# Display the results
display(sepsis3.head())